In [145]:
import duckdb
import pandas as pd
from pathlib import Path
import diskcache
from itertools import chain

import igraph as ig
import matplotlib.pyplot as plt
import seaborn.objects as so
import seaborn as sns

from utils.pandas_setup import pandas_setup
pandas_setup()

import pyalex
from pyalex import Works, Authors, Sources, Institutions, Topics, Publishers, Funders
pyalex.config.email = "Lawrence.Cram@anu.edu.au"
pyalex.config.max_retries = 0
pyalex.config.retry_backoff_factor = 0.1
pyalex.config.retry_http_codes = [429, 500, 503]

MY_DATA_PATH = Path('../DATA/')
MY_DATABASE_FILE = Path(MY_DATA_PATH / 'econ.duckdb')
MY_CACHE_FILE = Path('/home/lc/m/.cache/econommicsbusiness/cache.db')
DATAFILES_PATH = Path('../DATAFILES')

In [146]:
class SetUp:

    def __init__(self):
        self._setup_db()
        self._setup_cache()
        return
    
    def _setup_db(self):
        self.db = duckdb.connect(MY_DATABASE_FILE)
        self.db.sql("ATTACH IF NOT EXISTS ':memory:'")
        self.db.sql(""" SET memory_limit = '56GB';
                        SET threads = 6;
                        SET preserve_insertion_order = false;
                        SET order_by_non_integer_literal=true;
                        SET enable_progress_bar = true;
                        SET temp_directory = '/home/lc/m/.tmp';
                    """)
        self.db.sql("SHOW ALL TABLES").show()
        return
    
    def _setup_cache(self):
        self.cache = diskcache.Cache(MY_CACHE_FILE, size_limit=16_000_000_000)
        print(f'{self.cache.check() = }')
        print(f'{self.cache.volume() = }')
        return
    
    def name_of_global_obj(self, obj=None):
        for objname, oid in globals().items():
            if oid is obj:
                return objname
            

In [147]:
class CoauthorshipNetwork(SetUp):

    def __init__(self):
        super().__init__()
        return
    
    def build_authorship_network(self):
        # get works from a few small, top  journals
        source_list = 'https://openalex.org/S2764736659 https://openalex.org/S7397502'
        sql = f"""
                CREATE OR REPLACE TABLE memory.network AS 
                    SELECT replace(source_id, 'https://openalex.org/', '') AS source_id,
                            -- any_value(source_name),
                            replace(institution_id, 'https://openalex.org/', '') AS institution_id,
                            -- any_value(institution_name)
                            count(work_id) AS weight,
                        FROM works w
                        LEFT JOIN authorships a
                            USING (work_id) 
                            WHERE contains('{source_list}', w.source_id) = true 
                                    AND author_name NOT NULL
                                    AND work_id NOT NULL
                        GROUP BY source_id, institution_id, source_name, institution_name
                        ORDER BY weight DESC, source_name, institution_name
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.network").show()
        self.db.sql("COPY memory.network TO '../DATA/NETWORK_DATA/trial.csv' (HEADER false, DELIMITER '\t')")
        return
    
    def extract_graph(self):
        self.g = ig.Graph.Read_Ncol('../DATA/NETWORK_DATA/trial.csv', names=True)
        print(self.g.get_edgelist())
        print(self.g.es["weight"])
        print(self.g.vs["name"])
        return
    
    def pagerank(self):
        ranks = self.g.pagerank()
        print(ranks)
        return


In [ ]:
class CitationNetwork(SetUp):

    def __init__(self, kind=None):
        super().__init__()
        self.kind = kind
        return
    
    def build_citation_network(self):
        self._build_base()
        self._build_links()
        return

    def _build_base(self):
        sql = """
                CREATE OR REPLACE TABLE memory.base AS
                    SELECT DISTINCT w.work_id,
                            w.source_id,
                            w.source_name,
                            a.author_id,
                            a.author_name,
                            a.institution_id,
                            a.institution_name 
                        FROM works w
                            LEFT JOIN authorships a
                                USiNG (work_id)
                        WHERE contains('article review preprint letter', w.type) = true 
                    ORDER BY work_id
                    -- LIMIT 100000
            """
        self.db.sql(sql)
        self.db.sql("SELECT * FROM memory.base").show()
        return
    
    def _build_links(self):

        sql = f"""
                CREATE OR REPlACE TABLE memory.citation_network AS
                    SELECT -- sub.work_id as citer_id,
                            citer,
                            -- cited_id,
                            b2.{self.kind}_id AS cited,
                            count(sub.citer_id) AS weights
                        FROM
                        (
                        SELECT c.work_id AS citer_id, 
                                unnest(referenced_works) AS cited_id,
                                b1.{self.kind}_id AS citer,
                                b1.source_id,
                                b1.source_name,
                                b1.institution_id,
                                b1.institution_name,
                            FROM cited c
                            LEFT JOIN memory.base b1
                                ON c.work_id = b1.work_id
                        ) sub
                            LEFT JOIN memory.base b2
                                ON sub.cited_id = b2.work_id
                        WHERE citer NOT NULL 
                                AND b2.{self.kind}_id NOT NULL
                    GROUP BY ALL 
            """
        self.db.sql(sql)
        self.db.sql(f"COPY memory.citation_network TO '../DATA/NETWORK_DATA/citation_network_{self.kind}.csv' (HEADER true, DELIMITER ',')")
        return        
    
    def extract_graph(self):
        df_edges = pd.read_csv(f'../DATA/NETWORK_DATA/citation_network_{self.kind}.csv')
        df_edges = df_edges[df_edges.weights >= 50]
        print(f'{df_edges.shape = }\n{df_edges.head()}')
        self.g = ig.Graph.DataFrame(df_edges, directed=True, use_vids=False)
        print(self.g.get_edgelist())
        summary = ig.summary(self.g, verbosity=0, width=78, edge_list_format='auto', max_rows=99999, print_graph_attributes=False, 
                                          print_vertex_attributes=False, print_edge_attributes=False, full=False)
        print(summary)
        return
    
    def pagerank(self):
        ranks = [100.0*r for r in self.g.pagerank(weights='weights', implementation='power')]
        # n_ranks = len(ranks)+1
        # ranks = [(n_ranks-i)/float(n_ranks) for i in range(n_ranks)]
        temp = pd.DataFrame(zip(ranks, self.g.vs["name"], self.g.degree()), columns=['pageRank','citer', 'pub count']).sort_values('pageRank', ascending=False)
        if self.kind == 'source':
            pagerank = self.db.sql("SELECT DISTINCT p.*, source_name, host_name FROM temp p LEFT JOIN works w ON p.citer = w.source_id ORDER BY pageRank DESC").df()
        else:
            pagerank = self.db.sql("SELECT DISTINCT p.*, institution_name, country_code FROM temp p LEFT JOIN authorships a ON p.citer = a.institution_id ORDER BY pageRank DESC").df()
        self.db.sql(f"CREATE OR REPLACE TABLE econ.pagerank_{self.kind} AS SELECT * FROM pagerank")
        self.db.sql(f"SELECT * FROM econ.pagerank_{self.kind}").show()
        print(f'{pagerank.shape = }\n{pagerank.head(32)}')
        print(f'\n{pagerank.tail(32)}')
        print(f'Total publications {pagerank['pub count'].sum() = }')
        print(f'Sum of pageranks {pagerank['pageRank'].sum() = }')
        self.db.sql("SELECT count(DISTINCT work_id) AS original_works_count FROM works").show()
        return

    def weighted_citation_count(self):
        sql = f"""
                CREATE OR REPlACE TABLE econ.citation_counts AS
                    SELECT count(b2.work_id) AS citation_counts,
                            sum(p.pagerank) AS weighted_citation_counts,
                            b2.author_id AS cited_id,
                            b2.author_name AS cited_name,
                        FROM
                        (
                        SELECT c.work_id AS citer_id, 
                                unnest(referenced_works) AS cited_id,
                                b1.{self.kind}_id AS citer,
                                b1.source_id,
                                b1.institution_id
                            FROM cited c
                            LEFT JOIN memory.base b1
                                ON c.work_id = b1.work_id
                        ) sub
                            LEFT JOIN memory.base b2
                                ON sub.cited_id = b2.work_id
                                LEFT JOIN pagerank_{self.kind} p
                                    ON sub.{self.kind}_id = p.citer
                        WHERE p.citer NOT NULL AND b2.{self.kind}_id NOT NULL AND b2.author_id NOT NULL
                    GROUP BY ALL
                    ORDER BY citation_counts DESC
            """ 
        print('econ.citation_counts')
        self.db.sql(sql)
        self.db.sql("SELECT * FROM econ.citation_counts").show()
        return
    
    def plot_comparision(self):
        df = self.db.sql("SELECT * FROM econ.citation_counts").df().sort_values('weighted_citation_counts', ascending=False)
        sample = self.db.sql("SELECT * FROM econ.sample_names").df()
        dd = dict(zip(sample.author_id, sample.Group))
        print(f'{dd = }')
        df['group'] = [dd.get(aid, 'X') for aid in df.cited_id]
        df['pointsize'] = [2 if g == 'X' else 10 for g in df.group]
        df = df.sort_values('group', ascending=False)
        print(f'{sample.shape = }\n{sample.head()}')

        fig, ax = plt.subplots(1, 1)
        sns.scatterplot(df, x='citation_counts', y='weighted_citation_counts', hue='group', size='pointsize')
        ax.set_yscale('log')
        ax.set_xscale('log')
        ax.set_xlim((100, 50000))
        ax.set_ylim((100, 50000))
        hand, lab = ax.get_legend_handles_labels()
        lab[0] = 'Group'
        ax.legend()
        ax.legend(handles=[h for h in hand[:4]], 
                  labels=[l for l in lab[:4]],
                  loc=2)
        ax.set_title(f'{self.kind.title()}s citing {self.kind}s')
        plt.show()


        # p.theme({'legend.loc': 'center left'}).show()
        return
    
    def plot_distributions(self):
        df = self.db.sql("SELECT * FROM econ.pagerank_source").df()
        print(f'{df.shape = }\n{df.head()}')
        return

In [149]:
def main():

    # cn = CoauthorshipNetwork()
    # cn.build_authorship_network()
    # cn.extract_graph()
    # cn.pagerank()

    # kind = 'source'
    kind = 'institution'
    cn = CitationNetwork(kind=kind)
    cn.build_citation_network()
    cn.extract_graph()
    cn.pagerank()
    cn.weighted_citation_count()
    cn.plot_comparision()
    cn.plot_distributions()
    return

In [150]:
if __name__ == "__main__":
    main()
    print("DONE!")

┌──────────┬─────────┬──────────────────────┬──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────

BinderException: Binder Error: WHERE clause cannot contain aggregates!

LINE 7:                             count(sub.citer_id) AS weights
                                    ^